In [2]:
###Rule-Based Movie Recommendation Algorithm

import tkinter as tk
from tkinter import ttk, scrolledtext, messagebox
import pandas as pd
import re
from datetime import datetime
import threading
import time
import random
from urllib.parse import quote

class MovieRecommendationChatbot:
    def __init__(self, root):
        self.root = root
        self.root.title("🎬 CineBot - AI Movie Recommendation Assistant")
        self.root.geometry("1000x700")
        self.root.configure(bg='#0a0a0a')
        self.root.resizable(True, True)
        
        # Initialize variables
        self.movies_df = None
        self.conversation_history = []
        self.is_typing = False
        self.message_count = 0
        self.placeholder_active = True
        
        # Setup GUI first
        self.setup_enhanced_gui()
        
        # Auto-load dataset
        self.auto_load_dataset()

    def auto_load_dataset(self):
        """Load dataset.csv directly"""
        try:
            self.movies_df = pd.read_csv('dataset.csv')
            print(f"✅ Dataset loaded: {len(self.movies_df)} movies")
            
            # Update status bar
            self.status_label.config(text=f"🎬 {len(self.movies_df):,} movies loaded | Ready to help!")
            
            # Show welcome message
            self.root.after(1000, self.show_welcome_message)
            
        except FileNotFoundError:
            error_msg = "❌ dataset.csv not found in current directory. Please ensure the file exists."
            messagebox.showerror("Dataset Error", error_msg)
            self.status_label.config(text="❌ No dataset loaded", fg='#ff6b6b')
            
        except Exception as e:
            error_msg = f"❌ Error loading dataset.csv: {str(e)}"
            messagebox.showerror("Dataset Error", error_msg)
            self.status_label.config(text="❌ Dataset error", fg='#ff6b6b')

    def setup_enhanced_gui(self):
        """Setup enhanced modern GUI"""
        # Main container
        main_frame = tk.Frame(self.root, bg='#0a0a0a')
        main_frame.pack(fill=tk.BOTH, expand=True)
        
        # Header
        self.setup_header(main_frame)
        
        # Status bar (put at bottom first)
        self.setup_status_bar(main_frame)
        
        # Input area (put before chat so it's always visible)
        self.setup_input_area(main_frame)
        
        # Chat area (fill remaining space)
        self.setup_chat_area(main_frame)

    def setup_header(self, parent):
        """Setup modern header"""
        header_frame = tk.Frame(parent, bg='#1a1a2e', height=80)
        header_frame.pack(fill=tk.X)
        header_frame.pack_propagate(False)
        
        # Title
        title_label = tk.Label(
            header_frame, 
            text="🎬 CineBot", 
            font=('Segoe UI', 24, 'bold'), 
            fg='#ffffff', 
            bg='#1a1a2e'
        )
        title_label.pack(pady=(15, 0))
        
        # Subtitle
        subtitle_label = tk.Label(
            header_frame, 
            text="AI-Powered Movie Recommendation Assistant", 
            font=('Segoe UI', 10), 
            fg='#888888', 
            bg='#1a1a2e'
        )
        subtitle_label.pack(pady=(0, 15))

    def setup_chat_area(self, parent):
        """Setup chat area"""
        # Chat container - fill remaining space
        chat_container = tk.Frame(parent, bg='#0a0a0a')
        chat_container.pack(fill=tk.BOTH, expand=True, padx=20, pady=(10, 0))
        
        # Chat display
        self.chat_display = scrolledtext.ScrolledText(
            chat_container,
            wrap=tk.WORD,
            font=('Segoe UI', 11),
            bg='#1e1e1e',
            fg='#ffffff',
            insertbackground='#ffffff',
            selectbackground='#0078d4',
            selectforeground='#ffffff',
            relief=tk.FLAT,
            bd=0,
            padx=20,
            pady=20,
            spacing1=8,
            spacing2=4,
            spacing3=8
        )
        self.chat_display.pack(fill=tk.BOTH, expand=True)
        self.chat_display.config(state=tk.DISABLED)
        
        # Configure text tags
        self.setup_text_tags()

    def setup_text_tags(self):
        """Setup text formatting tags"""
        self.chat_display.tag_config("user_name", foreground="#00d4aa", font=('Segoe UI', 11, 'bold'), spacing1=10)
        self.chat_display.tag_config("user_message", foreground="#ffffff", font=('Segoe UI', 11), lmargin1=20, lmargin2=20, spacing3=10)
        self.chat_display.tag_config("bot_name", foreground="#0078d4", font=('Segoe UI', 11, 'bold'), spacing1=10)
        self.chat_display.tag_config("bot_message", foreground="#e6e6e6", font=('Segoe UI', 11), lmargin1=20, lmargin2=20, spacing3=10)
        self.chat_display.tag_config("movie_title", foreground="#ffd700", font=('Segoe UI', 11, 'bold'))
        self.chat_display.tag_config("rating", foreground="#ff6b6b", font=('Segoe UI', 10, 'bold'))
        self.chat_display.tag_config("genre", foreground="#4ecdc4", font=('Segoe UI', 10))
        self.chat_display.tag_config("typing", foreground="#888888", font=('Segoe UI', 10, 'italic'))

    def setup_input_area(self, parent):
        """Setup input area with fixed layout"""
        # Main input container - ensure it's always visible
        input_main_container = tk.Frame(parent, bg='#0a0a0a')
        input_main_container.pack(fill=tk.X, side=tk.BOTTOM, padx=20, pady=10)
        
        # Quick suggestions section
        suggestions_frame = tk.Frame(input_main_container, bg='#0a0a0a')
        suggestions_frame.pack(fill=tk.X, pady=(0, 10))
        
        suggestions_label = tk.Label(
            suggestions_frame,
            text="💡 Quick Suggestions:",
            font=('Segoe UI', 10, 'bold'),
            fg='#888888',
            bg='#0a0a0a'
        )
        suggestions_label.pack(anchor=tk.W, pady=(0, 5))
        
        # Suggestion buttons
        buttons_frame = tk.Frame(suggestions_frame, bg='#0a0a0a')
        buttons_frame.pack(fill=tk.X)
        
        suggestions = [
            ("🎭 Drama", "best drama movies"),
            ("😂 Comedy", "funny movies"),
            ("🎬 Action", "action movies"),
            ("⭐ Top Rated", "highest rated movies"),
            ("🆕 Recent", "recent movies"),
            ("🔍 Surprise", "recommend something good")
        ]
        
        for text, query in suggestions:
            btn = tk.Button(
                buttons_frame,
                text=text,
                command=lambda q=query: self.quick_search(q),
                font=('Segoe UI', 9),
                bg='#2d2d2d',
                fg='#ffffff',
                activebackground='#404040',
                activeforeground='#ffffff',
                relief=tk.FLAT,
                bd=0,
                padx=12,
                pady=6,
                cursor='hand2'
            )
            btn.pack(side=tk.LEFT, padx=(0, 8), pady=2)
            
            # Hover effects
            btn.bind('<Enter>', lambda e, b=btn: b.config(bg='#404040'))
            btn.bind('<Leave>', lambda e, b=btn: b.config(bg='#2d2d2d'))
        
        # Input section
        input_frame = tk.Frame(input_main_container, bg='#2d2d2d')
        input_frame.pack(fill=tk.X, pady=(10, 0))
        
        # Text input
        self.user_input = tk.Text(
            input_frame,
            height=3,
            font=('Segoe UI', 12),
            bg='#2d2d2d',
            fg='#ffffff',
            insertbackground='#ffffff',
            selectbackground='#0078d4',
            relief=tk.FLAT,
            bd=0,
            padx=15,
            pady=10,
            wrap=tk.WORD
        )
        self.user_input.pack(fill=tk.X, side=tk.LEFT, expand=True, padx=(15, 10), pady=10)
        self.user_input.bind('<Return>', self.handle_enter_key)
        self.user_input.bind('<KeyPress>', self.on_key_press)
        self.user_input.bind('<FocusIn>', self.on_focus_in)
        
        # Add placeholder
        self.add_placeholder()
        
        # Send button
        send_button = tk.Button(
            input_frame,
            text="Send\n✈️",
            command=self.send_message,
            font=('Segoe UI', 11, 'bold'),
            bg='#0078d4',
            fg='white',
            activebackground='#106ebe',
            activeforeground='white',
            relief=tk.FLAT,
            bd=0,
            width=8,
            cursor='hand2'
        )
        send_button.pack(side=tk.RIGHT, padx=(0, 15), pady=10)
        
        # Hover effects
        send_button.bind('<Enter>', lambda e: send_button.config(bg='#106ebe'))
        send_button.bind('<Leave>', lambda e: send_button.config(bg='#0078d4'))

    def setup_status_bar(self, parent):
        """Setup status bar"""
        status_frame = tk.Frame(parent, bg='#1a1a2e', height=30)
        status_frame.pack(fill=tk.X, side=tk.BOTTOM)
        status_frame.pack_propagate(False)
        
        self.status_label = tk.Label(
            status_frame,
            text="🔄 Loading dataset...",
            font=('Segoe UI', 9),
            fg='#ffaa00',
            bg='#1a1a2e'
        )
        self.status_label.pack(side=tk.LEFT, padx=15, pady=5)
        
        self.counter_label = tk.Label(
            status_frame,
            text="💬 0 messages",
            font=('Segoe UI', 9),
            fg='#888888',
            bg='#1a1a2e'
        )
        self.counter_label.pack(side=tk.RIGHT, padx=15, pady=5)

    def add_placeholder(self):
        """Add placeholder text"""
        placeholder_text = "Ask me about movies... (e.g., 'Best action movies', 'Tell me about Inception')"
        self.user_input.insert("1.0", placeholder_text)
        self.user_input.config(fg='#666666')
        self.placeholder_active = True

    def on_key_press(self, event):
        """Handle key press"""
        if self.placeholder_active:
            self.clear_placeholder()

    def on_focus_in(self, event):
        """Handle focus in"""
        if self.placeholder_active:
            self.clear_placeholder()

    def clear_placeholder(self):
        """Clear placeholder text"""
        if self.placeholder_active:
            self.user_input.delete("1.0", tk.END)
            self.user_input.config(fg='#ffffff')
            self.placeholder_active = False

    def show_welcome_message(self):
        """Show welcome message"""
        dataset_count = len(self.movies_df) if self.movies_df is not None else 0
        
        welcome_msg = f"""🎬 Welcome to CineBot! I have access to {dataset_count:,} movies from your dataset.

I can help you with:
✨ Movie recommendations by genre or mood
🔍 Information about specific movies  
⚖️ Movie comparisons
🏆 Top-rated films and rankings
📅 Movies from specific years
🎭 Personalized suggestions

Just type your question below or use the quick suggestions!"""
        
        self.add_bot_message(welcome_msg)

    def quick_search(self, query):
        """Handle quick search"""
        self.clear_placeholder()
        self.user_input.delete(1.0, tk.END)
        self.user_input.insert(1.0, query)
        self.send_message()

    def handle_enter_key(self, event):
        """Handle Enter key"""
        if event.state & 0x4:  # Ctrl+Enter for new line
            return
        else:
            self.send_message()
            return 'break'

    def send_message(self):
        """Send message"""
        user_text = self.user_input.get(1.0, tk.END).strip()
        if not user_text or self.placeholder_active:
            return
        
        # Update counter
        self.message_count += 1
        self.counter_label.config(text=f"💬 {self.message_count} messages")
        
        # Add user message
        self.add_user_message(user_text)
        self.user_input.delete(1.0, tk.END)
        
        # Show typing
        self.show_typing_indicator()
        
        # Process
        threading.Thread(target=self.process_user_message, args=(user_text,), daemon=True).start()

    def process_user_message(self, user_text):
        """Process user message"""
        time.sleep(random.uniform(0.8, 2.0))
        
        try:
            response = self.generate_smart_response(user_text)
            self.root.after(0, lambda: self.add_bot_message(response))
        except Exception as e:
            error_msg = f"❌ Error: {str(e)}"
            self.root.after(0, lambda: self.add_bot_message(error_msg))

    def generate_smart_response(self, user_message):
        """Generate response using actual dataset"""
        if self.movies_df is None or self.movies_df.empty:
            return "❌ No dataset available. Please ensure dataset.csv is in the current directory."
        
        user_message_lower = user_message.lower()
        
        # Specific movie search
        if any(phrase in user_message_lower for phrase in ['tell me about', 'what is', 'info about', 'details about']):
            return self.search_specific_movie(user_message)
        
        # Comparison
        if ' vs ' in user_message_lower or ' versus ' in user_message_lower:
            return self.compare_movies(user_message)
        
        # Top/best movies
        if any(word in user_message_lower for word in ['best', 'top', 'highest', 'greatest']):
            return self.get_top_movies(user_message)
        
        # Year-based
        if re.search(r'\b(19|20)\d{2}\b', user_message):
            return self.get_movies_by_year(user_message)
        
        # Similar movies
        if ' like ' in user_message_lower:
            return self.get_similar_movies(user_message)
        
        # General recommendations
        return self.get_general_recommendations(user_message)

    def search_specific_movie(self, user_message):
        """Search for specific movie"""
        patterns = [
            r'tell me about (.+)',
            r'what is (.+?) about',
            r'what is (.+)',
            r'info about (.+)',
            r'details about (.+)'
        ]
        
        movie_title = None
        for pattern in patterns:
            match = re.search(pattern, user_message.lower())
            if match:
                movie_title = match.group(1).strip()
                movie_title = re.sub(r'\b(the movie|movie|film)\b', '', movie_title).strip()
                break
        
        if not movie_title:
            return "🤔 Please specify which movie you'd like to know about."
        
        # Search in your actual dataset
        search_results = self.movies_df[
            self.movies_df['title'].str.contains(movie_title, case=False, na=False)
        ]
        
        if search_results.empty:
            return f"🔍 Couldn't find '{movie_title}' in the dataset. Try checking the spelling."
        
        movie = search_results.iloc[0]
        response = f"🎬 **{movie['title']}**\n\n"
        
        # Add available details from your dataset
        details = []
        
        if 'release_date' in movie and pd.notna(movie['release_date']):
            try:
                year = movie['release_date'].split('/')[-1] if '/' in str(movie['release_date']) else str(movie['release_date'])[:4]
                details.append(f"📅 {year}")
            except:
                pass
        
        if 'genre' in movie and pd.notna(movie['genre']):
            details.append(f"🎭 {movie['genre']}")
        
        if 'vote_average' in movie and pd.notna(movie['vote_average']):
            rating = movie['vote_average']
            stars = "⭐" * min(5, int(rating / 2))
            details.append(f"{stars} {rating}/10")
        
        if 'vote_count' in movie and pd.notna(movie['vote_count']):
            details.append(f"👥 {movie['vote_count']:,} votes")
        
        if details:
            response += " | ".join(details) + "\n\n"
        
        if 'overview' in movie and pd.notna(movie['overview']):
            response += f"📝 **Plot:**\n{movie['overview']}\n\n"
        
        response += "💡 Want similar recommendations or more details?"
        return response

    def get_top_movies(self, user_message):
        """Get top movies from dataset"""
        user_message_lower = user_message.lower()
        
        num_movies = 5
        if 'top 10' in user_message_lower or 'best 10' in user_message_lower:
            num_movies = 10
        elif 'top 3' in user_message_lower or 'best 3' in user_message_lower:
            num_movies = 3
        
        filtered_movies = self.movies_df.copy()
        
        # Genre filtering
        genre_keywords = ['drama', 'comedy', 'action', 'horror', 'romance', 'thriller', 'crime', 'animation', 'adventure', 'fantasy']
        for genre in genre_keywords:
            if genre in user_message_lower:
                if 'genre' in filtered_movies.columns:
                    filtered_movies = filtered_movies[
                        filtered_movies['genre'].str.contains(genre, case=False, na=False)
                    ]
                break
        
        # Sort by rating
        if 'vote_average' in filtered_movies.columns:
            top_movies = filtered_movies.nlargest(num_movies, 'vote_average')
        else:
            top_movies = filtered_movies.head(num_movies)
        
        if top_movies.empty:
            return "🔍 No movies found matching your criteria."
        
        response = f"🏆 Top {len(top_movies)} Movies:\n\n"
        
        for idx, (_, movie) in enumerate(top_movies.iterrows(), 1):
            response += f"{idx}. 🎬 **{movie['title']}**"
            
            if 'release_date' in movie and pd.notna(movie['release_date']):
                try:
                    year = str(movie['release_date']).split('/')[-1] if '/' in str(movie['release_date']) else str(movie['release_date'])[:4]
                    response += f" ({year})"
                except:
                    pass
            
            response += "\n"
            
            details = []
            if 'vote_average' in movie and pd.notna(movie['vote_average']):
                rating = movie['vote_average']
                stars = "⭐" * min(5, int(rating / 2))
                details.append(f"{stars} {rating}/10")
            
            if 'genre' in movie and pd.notna(movie['genre']):
                details.append(f"🎭 {movie['genre']}")
            
            if details:
                response += "   " + " | ".join(details) + "\n"
            
            if 'overview' in movie and pd.notna(movie['overview']):
                overview = str(movie['overview'])[:120] + "..."
                response += f"   📝 {overview}\n"
            
            response += "\n"
        
        return response

    def get_general_recommendations(self, user_message):
        """Get general recommendations"""
        user_message_lower = user_message.lower()
        filtered_movies = self.movies_df.copy()
        
        # Genre filtering
        if any(word in user_message_lower for word in ['comedy', 'funny', 'humor']):
            if 'genre' in filtered_movies.columns:
                filtered_movies = filtered_movies[
                    filtered_movies['genre'].str.contains('Comedy', case=False, na=False)
                ]
        elif any(word in user_message_lower for word in ['action', 'adventure']):
            if 'genre' in filtered_movies.columns:
                filtered_movies = filtered_movies[
                    filtered_movies['genre'].str.contains('Action', case=False, na=False)
                ]
        elif any(word in user_message_lower for word in ['drama', 'dramatic']):
            if 'genre' in filtered_movies.columns:
                filtered_movies = filtered_movies[
                    filtered_movies['genre'].str.contains('Drama', case=False, na=False)
                ]
        
        # Rating filter
        if any(word in user_message_lower for word in ['good', 'best', 'great']):
            if 'vote_average' in filtered_movies.columns:
                filtered_movies = filtered_movies[filtered_movies['vote_average'] >= 7.5]
        
        # Sort and get recommendations
        if 'vote_average' in filtered_movies.columns:
            recommendations = filtered_movies.nlargest(5, 'vote_average')
        else:
            recommendations = filtered_movies.head(5)
        
        if recommendations.empty:
            return "🤔 No movies found. Try different keywords!"
        
        response = "🎬 Here are my recommendations:\n\n"
        
        for idx, (_, movie) in enumerate(recommendations.iterrows(), 1):
            response += f"{idx}. 🌟 **{movie['title']}**\n"
            
            details = []
            if 'genre' in movie and pd.notna(movie['genre']):
                details.append(f"🎭 {movie['genre']}")
            
            if 'vote_average' in movie and pd.notna(movie['vote_average']):
                rating = movie['vote_average']
                stars = "⭐" * min(5, int(rating / 2))
                details.append(f"{stars} {rating}/10")
            
            if details:
                response += "   " + " | ".join(details) + "\n"
            
            if 'overview' in movie and pd.notna(movie['overview']):
                overview = str(movie['overview'])[:100] + "..."
                response += f"   📝 {overview}\n"
            
            response += "\n"
        
        return response

    def get_movies_by_year(self, user_message):
        """Get movies by year from dataset"""
        year_match = re.search(r'\b(19|20)\d{2}\b', user_message)
        if not year_match:
            return "🤔 Please specify a valid year."
        
        target_year = year_match.group()
        
        if 'release_date' not in self.movies_df.columns:
            return "📅 Release date information not available."
        
        try:
            year_movies = self.movies_df[
                self.movies_df['release_date'].astype(str).str.contains(target_year, na=False)
            ]
            
            if year_movies.empty:
                return f"🔍 No movies found for {target_year}."
            
            if 'vote_average' in year_movies.columns:
                year_movies = year_movies.nlargest(5, 'vote_average')
            else:
                year_movies = year_movies.head(5)
            
            response = f"🎬 Movies from {target_year}:\n\n"
            
            for idx, (_, movie) in enumerate(year_movies.iterrows(), 1):
                response += f"{idx}. 🌟 **{movie['title']}**\n"
                
                if 'vote_average' in movie and pd.notna(movie['vote_average']):
                    rating = movie['vote_average']
                    stars = "⭐" * min(5, int(rating / 2))
                    response += f"   {stars} {rating}/10\n"
                
                response += "\n"
            
            return response
            
        except Exception as e:
            return f"❌ Error: {str(e)}"

    def get_similar_movies(self, user_message):
        """Get similar movies"""
        like_match = re.search(r'like (.+?)(?:\?|$|,)', user_message.lower())
        if not like_match:
            return "🤔 Please specify which movie you'd like recommendations similar to."
        
        reference_movie = like_match.group(1).strip()
        
        ref_movies = self.movies_df[
            self.movies_df['title'].str.contains(reference_movie, case=False, na=False)
        ]
        
        if ref_movies.empty:
            return f"🔍 Couldn't find '{reference_movie}' in the dataset."
        
        ref_movie = ref_movies.iloc[0]
        similar_movies = self.movies_df.copy()
        
        if 'genre' in ref_movie and pd.notna(ref_movie['genre']):
            similar_movies = similar_movies[
                similar_movies['genre'].str.contains(ref_movie['genre'], case=False, na=False)
            ]
        
        similar_movies = similar_movies[similar_movies['title'] != ref_movie['title']]
        
        if 'vote_average' in similar_movies.columns:
            similar_movies = similar_movies.nlargest(5, 'vote_average')
        else:
            similar_movies = similar_movies.head(5)
        
        if similar_movies.empty:
            return f"🤔 No similar movies found to '{ref_movie['title']}'."
        
        response = f"🎬 Movies Similar to **{ref_movie['title']}**:\n\n"
        
        for idx, (_, movie) in enumerate(similar_movies.iterrows(), 1):
            response += f"{idx}. 🌟 **{movie['title']}**\n"
            
            if 'vote_average' in movie and pd.notna(movie['vote_average']):
                rating = movie['vote_average']
                stars = "⭐" * min(5, int(rating / 2))
                response += f"   {stars} {rating}/10\n"
            
            response += "\n"
        
        return response

    def compare_movies(self, user_message):
        """Compare movies from dataset"""
        parts = re.split(r'\s+vs\s+|\s+versus\s+', user_message.lower())
        if len(parts) < 2:
            return "🤔 Please specify two movies to compare."
        
        movie1_name = parts[0].strip()
        movie2_name = parts[1].strip()
        
        movie1 = self.movies_df[self.movies_df['title'].str.contains(movie1_name, case=False, na=False)]
        movie2 = self.movies_df[self.movies_df['title'].str.contains(movie2_name, case=False, na=False)]
        
        if movie1.empty or movie2.empty:
            return "🔍 One or both movies not found in dataset."
        
        m1 = movie1.iloc[0]
        m2 = movie2.iloc[0]
        
        response = f"⚖️ **{m1['title']}** vs **{m2['title']}**\n\n"
        
        if 'vote_average' in self.movies_df.columns:
            response += f"⭐ Ratings: {m1['vote_average']}/10 vs {m2['vote_average']}/10\n"
        
        if 'genre' in self.movies_df.columns:
            response += f"🎭 Genres: {m1['genre']} vs {m2['genre']}\n"
        
        return response

    def show_typing_indicator(self):
        """Show typing indicator"""
        self.is_typing = True
        timestamp = datetime.now().strftime("%H:%M")
        
        self.chat_display.config(state=tk.NORMAL)
        self.chat_display.insert(tk.END, f"\n🤖 CineBot [{timestamp}]\n", "bot_name")
        self.chat_display.insert(tk.END, "💭 Searching movies...\n", "typing")
        self.chat_display.config(state=tk.DISABLED)
        self.chat_display.see(tk.END)

    def add_user_message(self, message):
        """Add user message"""
        timestamp = datetime.now().strftime("%H:%M")
        
        self.chat_display.config(state=tk.NORMAL)
        self.chat_display.insert(tk.END, f"\n👤 You [{timestamp}]\n", "user_name")
        self.chat_display.insert(tk.END, f"{message}\n", "user_message")
        self.chat_display.config(state=tk.DISABLED)
        self.chat_display.see(tk.END)

    def add_bot_message(self, message):
        """Add bot message"""
        timestamp = datetime.now().strftime("%H:%M")
        
        self.chat_display.config(state=tk.NORMAL)
        
        # Remove typing indicator
        if self.is_typing:
            content = self.chat_display.get(1.0, tk.END)
            if "💭 Searching movies..." in content:
                lines = content.split('\n')
                new_content = []
                for i, line in enumerate(lines):
                    if "💭 Searching movies..." in line:
                        if new_content and "🤖 CineBot" in new_content[-1]:
                            new_content.pop()
                        break
                    new_content.append(line)
                
                self.chat_display.delete(1.0, tk.END)
                self.chat_display.insert(1.0, '\n'.join(new_content))
            
            self.is_typing = False
        
        self.chat_display.insert(tk.END, f"\n🤖 CineBot [{timestamp}]\n", "bot_name")
        self.insert_formatted_message(message)
        self.chat_display.config(state=tk.DISABLED)
        self.chat_display.see(tk.END)

    def insert_formatted_message(self, message):
        """Insert formatted message"""
        lines = message.split('\n')
        
        for line in lines:
            if '**' in line:
                parts = line.split('**')
                for i, part in enumerate(parts):
                    if i % 2 == 0:
                        self.chat_display.insert(tk.END, part, "bot_message")
                    else:
                        self.chat_display.insert(tk.END, part, "movie_title")
            elif '⭐' in line:
                self.chat_display.insert(tk.END, line, "rating")
            elif '🎭' in line:
                self.chat_display.insert(tk.END, line, "genre")
            else:
                self.chat_display.insert(tk.END, line, "bot_message")
            
            self.chat_display.insert(tk.END, '\n', "bot_message")

def main():
    root = tk.Tk()
    app = MovieRecommendationChatbot(root)
    root.mainloop()

if __name__ == "__main__":
    main()